# IMGS thuần: fine-tune độc lập cho từng test pose

Mỗi pose luôn load lại `iteration_30000`, chấm toàn bộ train view bằng
`exp(-d² / (2(3σ)²)) * max(0, cosθ)²`, chọn top-25, rồi fine-tune đúng
3.000 optimizer step. LAS/EAS/RAP/MU của ImprovedGS vẫn bật; coarse-to-fine
và pose-aware sampling đều tắt. Densify/split chỉ chạy trong 1.500 step đầu.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_DIR = Path('/kaggle/working/Improved-GS')
SCENE_NAME = 'HCM0204'
SOURCE_PATH = Path('/kaggle/working/vai_native_simple_radial/public_set') / SCENE_NAME

# Tự tìm đúng cấu trúc model trong ảnh. Nếu có nhiều kết quả, đặt BASE_MODEL thủ công.
BASE_MODEL_CANDIDATES = sorted(Path('/kaggle/input').glob(f'**/vai_models/{SCENE_NAME}'))
if len(BASE_MODEL_CANDIDATES) != 1:
    raise RuntimeError(f'Cần đúng 1 base model, tìm được: {BASE_MODEL_CANDIDATES}')
BASE_MODEL = BASE_MODEL_CANDIDATES[0]

OUTPUT_MODEL_ROOT = Path('/kaggle/working/vai_test_pose_models/public_set') / SCENE_NAME
RENDER_ROOT = Path('/kaggle/working/vai_test_pose_submission/public_set')
PNG_ROOT = Path('/kaggle/working/vai_test_pose_png/public_set')
BASE_ITERATION = 30_000
FINE_TUNE_STEPS = 3_000
SPLIT_UNTIL_STEP = 1_500
TOP_K = 25
SIGMA_MULTIPLIER = 3.0
DENSIFY_GRAD_THRESHOLD = 0.0002
SAVE_POSE_MODELS = False  # True se ton rat nhieu dung luong voi model 5.5M GS

base_ply = BASE_MODEL / 'point_cloud' / f'iteration_{BASE_ITERATION}' / 'point_cloud.ply'
assert (REPO_DIR / 'vai_test_pose_finetune.py').is_file(), REPO_DIR
assert SOURCE_PATH.is_dir(), SOURCE_PATH
assert base_ply.is_file(), base_ply

base_parameters_path = BASE_MODEL / 'training_parameters.json'
base_parameters = json.loads(base_parameters_path.read_text()) if base_parameters_path.is_file() else {}
BASE_BUDGET = int(base_parameters.get('budget', 5_500_000))
print('Source:', SOURCE_PATH)
print('Base model:', BASE_MODEL)
print('Base budget:', BASE_BUDGET)
print('Output:', OUTPUT_MODEL_ROOT)

In [ ]:
command = [
    sys.executable, 'vai_test_pose_finetune.py',
    '--source_path', str(SOURCE_PATH),
    '--base_model_path', str(BASE_MODEL),
    '--base_iteration', str(BASE_ITERATION),
    '--model_path', str(OUTPUT_MODEL_ROOT),
    '--scene_name', SCENE_NAME,
    '--fine_tune_steps', str(FINE_TUNE_STEPS),
    '--split_from_step', '0',
    '--split_until_step', str(SPLIT_UNTIL_STEP),
    '--top_k', str(TOP_K),
    '--sigma_multiplier', str(SIGMA_MULTIPLIER),
    '--save_pose_models', str(SAVE_POSE_MODELS).lower(),
    '--training_method', 'improvedgs',
    '--use_las', 'true',
    '--use_eas', 'true',
    '--use_rap', 'true',
    '--use_mu', 'true',
    '--coarse_to_fine', 'false',
    '--pose_aware_sampling', 'false',
    '--densify_grad_threshold', str(DENSIFY_GRAD_THRESHOLD),
    '--budget', str(BASE_BUDGET),
    '--resolution', '-1',
    '--data_device', 'cpu',
    '--eval', 'false',
    '--train_test_exp', 'false',
    '--output_root', str(RENDER_ROOT),
    '--output_extension', 'csv',
    '--save_png', 'true',
    '--png_root', str(PNG_ROOT),
    '--evaluate', 'true',
    '--require_gt', 'false',
    '--overwrite', 'true',
    '--progress_bar_width', '100',
]
print(' '.join(command))
subprocess.run(command, cwd=REPO_DIR, check=True)

In [ ]:
manifest_path = OUTPUT_MODEL_ROOT / 'test_pose_finetune_manifest.json'
manifest = json.loads(manifest_path.read_text())
print('Completed:', manifest['completed_pose_count'], '/', manifest['test_pose_count'])
print('Render dir:', manifest['render_dir'])
if 'evaluation' in manifest:
    print(json.dumps(manifest['evaluation'], indent=2))

# Mỗi pose luôn có record và bảng score/top-25 riêng; PLY chỉ có khi SAVE_POSE_MODELS=True.
first_pose = manifest['poses'][0]
print('First pose record:', first_pose['model_path'])
print('First pose PLY:', first_pose['point_cloud_path'] or '(khong luu)')
print('First pose top view:', first_pose['selection']['views'][0])